## Qlib 工作流中文注释与导航（为初学者准备）

以下是对本笔记本主要步骤、关键对象与常用参数的中文说明。阅读/运行时建议从上到下依次进行，并在每一步完成后观察输出与记录文件。

---

### 1. 初始化与数据准备
- `qlib.init(provider_uri=..., region=REG_CN)`: 初始化 Qlib 客户端。
  - `provider_uri`: 本地数据目录（如 `~/.qlib/qlib_data/cn_data`）。若首次使用，请先下载数据（参见 `scripts/README.md` 或 `scripts/get_data.py`）。
  - `region`: 区域设置（如 `REG_CN` 表示中国 A 股）。影响交易日、时区、撮合规则等。
- 运行前请确认：本地数据路径存在且与 `provider_uri` 保持一致；Windows 下 `~` 会展开到用户目录。

### 2. 市场与基准设置
- `market`: 股票池标识（如 `"csi300"` 对应沪深300成分）。
- `benchmark`: 回测对比基准（如 `"SH000300"` 表示沪深300指数）。
- 更换股票池/基准时，请确保数据目录中有对应成分与指数。

### 3. 任务配置（Task = 模型 + 数据集）
Qlib 将一次训练-预测-回测流程抽象为一个“任务（task）”，包含 `model` 与 `dataset` 两部分。

- 数据集 `dataset`（通常使用 `DatasetH`）
  - `handler`: 特征与标签的生成器。
    - 常见为 `Alpha158`（日频 158 特征，内置标签定义），也可替换为自定义 Handler。
  - `kwargs` 关键时间参数：
    - `start_time`, `end_time`: 允许读取/缓存数据的整体窗口。
    - `fit_start_time`, `fit_end_time`: 用于拟合特征归一化等统计量的时间窗口（只在训练期拟合，避免信息泄露）。
    - `instruments`: 股票池（通常与 `market` 一致）。
  - `segments` 切分：
    - `train`, `valid`, `test`: 训练/验证/测试区间，建议满足“先训练、再验证、最后测试”的时间顺序，且都应在 `start~end` 窗口内。

- 模型 `model`（如 `LGBModel`）
  - 常用超参：
    - `learning_rate`: 学习率（越小越稳，但需更多迭代）。
    - `num_leaves`, `max_depth`: 模型容量（越大越容易过拟合）。
    - `subsample`, `colsample_bytree`: 行/列采样比例（有助泛化）。
    - `lambda_l1`, `lambda_l2`: 正则化强度。
    - `num_threads`: 线程数（可按 CPU 核数设置）。

提示：时间关系建议是 `start/end` 覆盖 `train/valid/test` 的外包络；`fit_*` 至少覆盖训练段。

### 4. 实验记录与训练
- `R.start(experiment_name=...)`: 打开一次实验上下文（便于追踪与复现）。
- `R.log_params(...)`: 记录本次配置（参数会保存到记录目录中）。
- `model.fit(dataset)`: 在训练集上训练，并利用验证集做早停/调参（具体依模型而定）。
- `R.save_objects(trained_model=model)`: 保存训练好的模型快照。
- `rid = R.get_recorder().id`: 获取记录器 ID，后续预测/回测可复用。

建议：每次改动参数后都开启新的实验名，便于横向对比。

### 5. 预测信号、回测与分析
- 生成信号：`SignalRecord(model, dataset, recorder).generate()`
  - 会在指定分段（通常是 `test`）上生成预测得分并保存为 `pred.pkl`。
- 回测与组合分析：`PortAnaRecord(recorder, port_analysis_config, freq).generate()`
  - `executor`：撮合执行配置（如 `time_per_step="day"` 日频；`generate_portfolio_metrics=True` 生成组合指标）。
  - `strategy`：交易策略（常见 `TopkDropoutStrategy`）。
    - 关键参数：`topk`（每期持仓数）、`n_drop`（每期剔除的最差持仓数，控制换手）。
  - `backtest`：回测配置
    - `start_time`, `end_time`: 回测区间（通常与 `test` 一致）。
    - `account`: 初始资金。
    - `benchmark`: 基准指数。
    - `exchange_kwargs`: 交易撮合细节，如 `freq`、`limit_threshold`（涨跌停限制）、`deal_price`（成交价假设，常用 `close`）、`open_cost/close_cost/min_cost`（手续费）。

提示：成交价假设、成本与涨跌停设置会显著影响实际回测表现。

### 6. 常见结果与可视化
- 常见产出文件：
  - `pred.pkl`: 每期股票的预测分数。
  - `portfolio_analysis/report_normal_1day.pkl`: 组合收益统计。
  - `positions_normal_1day.pkl`: 每期持仓明细。
  - `port_analysis_1day.pkl`: 风险指标与曲线。
- 常用图表：
  - 收益/回撤报告：`analysis_position.report_graph(...)`
  - 风险分析：`analysis_position.risk_analysis_graph(...)`
  - 预测效果（IC/分层）：`analysis_position.score_ic_graph(...)`，`analysis_model.model_performance_graph(...)`

### 7. 快速调参指南（常改项）
- 数据切分：修改 `segments.train/valid/test` 的起止日期。
- 股票池/基准：调整 `market` 与 `benchmark`，确保数据存在。
- 特征与标签：将 `handler.class` 由 `Alpha158` 改为其他 Handler 或自定义实现。
- 模型容量与正则：调 `num_leaves`、`max_depth`、`lambda_l1/lambda_l2`；必要时使用交叉验证。
- 策略与换手：调 `topk`、`n_drop`，或替换为不同策略组件。
- 交易假设：调 `deal_price`、手续费与涨跌停，评估鲁棒性。

### 8. 复现实验与对比
- 通过 `experiment_name` 与 `recorder id` 可精确定位某次训练与其产物。
- 建议为每次关键改动建立新实验名；运行结束后保留 `rid` 以便复用同一模型做回测与分析。

---

有任何段落不清晰，或你想切换到分钟级数据/自定义特征，我可以在相应代码单元处加上针对性的中文行内注释与最小修改示例。


In [ ]:
#  Copyright (c) Microsoft Corporation.
#  Licensed under the MIT License.

In [ ]:
import sys, site
from pathlib import Path

################################# NOTE #################################
#  Please be aware that if colab installs the latest numpy and pyqlib  #
#  in this cell, users should RESTART the runtime in order to run the  #
#  following cells successfully.                                       #
########################################################################

# try:
#     import qlib
# except ImportError:
#     # install qlib
#     ! pip install --upgrade numpy
#     ! pip install pyqlib
#     if "google.colab" in sys.modules:
#         # The Google colab environment is a little outdated. We have to downgrade the pyyaml to make it compatible with other packages
#         ! pip install pyyaml==5.4.1
#     # reload
#     site.main()

scripts_dir = Path.cwd().parent.joinpath("scripts")
if not scripts_dir.joinpath("get_data.py").exists():
    # download get_data.py script
    scripts_dir = Path("~/tmp/qlib_code/scripts").expanduser().resolve()
    scripts_dir.mkdir(parents=True, exist_ok=True)
    import requests

    with requests.get("https://raw.githubusercontent.com/microsoft/qlib/main/scripts/get_data.py", timeout=10) as resp:
        with open(scripts_dir.joinpath("get_data.py"), "wb") as fp:
            fp.write(resp.content)

In [ ]:
import qlib
import pandas as pd
from qlib.constant import REG_CN
from qlib.utils import exists_qlib_data, init_instance_by_config
from qlib.workflow import R
from qlib.workflow.record_temp import SignalRecord, PortAnaRecord
from qlib.utils import flatten_dict

In [ ]:
# 使用本地默认数据目录；若不存在则自动下载
# NOTE: 也可手动执行：python scripts/get_data.py qlib_data_cn --target_dir ~/.qlib/qlib_data/cn_data
provider_uri = "./.qlib/qlib_data/cn_data"  # 目标数据目录（可按需修改）
# if not exists_qlib_data(provider_uri):
#     print(f"Qlib data is not found in {provider_uri}")
#     sys.path.append(str(scripts_dir))  # 将脚本目录加入路径，便于导入 get_data
#     from get_data import GetData

#     # 下载中国市场日频数据到指定目录
#     GetData().qlib_data(target_dir=provider_uri, region=REG_CN)

# 初始化 Qlib（区域设置影响交易日、撮合等）
qlib.init(provider_uri=provider_uri, region=REG_CN)

In [ ]:
# 股票池（如沪深300成分）
market = "csi300"
# 回测基准指数（沪深300指数）
benchmark = "SH000300"

# 训练模型（train model）

下方单元将配置特征/标签的数据集与 LightGBM 模型，并启动一次实验记录后进行训练。

In [ ]:
###################################
# 训练配置与模型训练（train model）
###################################
# 数据处理配置：用于特征归一化统计等（避免信息泄露，fit_* 仅覆盖训练期）
data_handler_config = {
    "start_time": "2008-01-01",
    "end_time": "2020-08-01",
    "fit_start_time": "2008-01-01",
    "fit_end_time": "2014-12-31",
    "instruments": market,  # 股票池（与上文 market 保持一致）
}

# Task 由模型与数据集组成
task = {
    "model": {
        "class": "LGBModel",
        "module_path": "qlib.contrib.model.gbdt",
        "kwargs": {
            "loss": "mse",               # 回归损失
            "colsample_bytree": 0.8879,  # 列采样比例（防过拟合）
            "learning_rate": 0.0421,     # 学习率
            "subsample": 0.8789,         # 行采样比例
            "lambda_l1": 205.6999,       # L1 正则
            "lambda_l2": 580.9768,       # L2 正则
            "max_depth": 8,              # 最大深度（控制模型容量）
            "num_leaves": 210,           # 叶子数（控制模型容量）
            "num_threads": 20,           # 线程数
        },
    },
    "dataset": {
        "class": "DatasetH",
        "module_path": "qlib.data.dataset",
        "kwargs": {
            "handler": {
                "class": "Alpha158",                  # 内置 158 因子特征
                "module_path": "qlib.contrib.data.handler",
                "kwargs": data_handler_config,
            },
            "segments": {                              # 数据集切分
                "train": ("2008-01-01", "2014-12-31"),
                "valid": ("2015-01-01", "2016-12-31"),
                "test": ("2017-01-01", "2020-08-01"),
            },
        },
    },
}

# 初始化实例
model = init_instance_by_config(task["model"])   # 构建 LGB 模型
dataset = init_instance_by_config(task["dataset"]) # 构建数据集

# 启动实验记录并训练
with R.start(experiment_name="train_model"):
    R.log_params(**flatten_dict(task))  # 记录参数，便于复现
    model.fit(dataset)                  # 训练（含验证早停/评估）
    R.save_objects(trained_model=model) # 保存模型
    rid = R.get_recorder().id           # 记录器 ID（后续复用）

# 预测、回测与分析（prediction, backtest & analysis）

将使用上一步训练好的模型在测试集生成预测信号，并基于简单的 TopK 策略进行日频回测与组合分析。

In [ ]:
###################################
# 预测信号、回测与组合分析（prediction, backtest & analysis）
###################################
# 回测与执行配置：包含撮合、策略与回测窗口、成本等
port_analysis_config = {
    "executor": {
        "class": "SimulatorExecutor",
        "module_path": "qlib.backtest.executor",
        "kwargs": {
            "time_per_step": "day",              # 日频回测
            "generate_portfolio_metrics": True,   # 生成组合指标
        },
    },
    "strategy": {
        "class": "TopkDropoutStrategy",
        "module_path": "qlib.contrib.strategy.signal_strategy",
        "kwargs": {
            "model": model,        # 使用训练好的模型打分
            "dataset": dataset,    # 使用同一数据集对象（含 test 切分）
            "topk": 50,            # 每期持仓数
            "n_drop": 5,           # 每期剔除最差持仓数（控制换手）
        },
    },
    "backtest": {
        "start_time": "2017-01-01",
        "end_time": "2020-08-01",
        "account": 100000000,       # 初始资金
        "benchmark": benchmark,     # 基准指数
        "exchange_kwargs": {
            "freq": "day",               # 交易频率
            "limit_threshold": 0.095,     # 涨跌停（±9.5%）
            "deal_price": "open",        # 成交价假设（可改为 close）
            "open_cost": 0.0005,          # 买入手续费率
            "close_cost": 0.0015,         # 卖出手续费率
            "min_cost": 5,                # 最低手续费
        },
    },
}

# 回测并记录结果
with R.start(experiment_name="backtest_analysis"):
    recorder = R.get_recorder(recorder_id=rid, experiment_name="train_model")
    model = recorder.load_object("trained_model")

    # 预测信号并保存（pred.pkl）
    recorder = R.get_recorder()
    ba_rid = recorder.id
    sr = SignalRecord(model, dataset, recorder)
    sr.generate()

    # 回测与组合分析（生成报告与持仓等 pkl）
    par = PortAnaRecord(recorder, port_analysis_config, "day")
    par.generate()

# 可视化分析图表（analyze graphs）

本节从记录中加载回测结果与预测分数，绘制收益/风险与 IC 等图表。

In [ ]:
from qlib.contrib.report import analysis_model, analysis_position
from qlib.data import D

# 加载回测阶段的记录器与产物（预测分数、组合报告、持仓、风险分析）
recorder = R.get_recorder(recorder_id=ba_rid, experiment_name="backtest_analysis")
print(recorder)
# 预测得分（按时间×标的的 MultiIndex）
pred_df = recorder.load_object("pred.pkl")
# 组合收益/回撤等统计
report_normal_df = recorder.load_object("portfolio_analysis/report_normal_1day.pkl")
# 每期持仓明细
positions = recorder.load_object("portfolio_analysis/positions_normal_1day.pkl")
# 风险指标与时间序列
analysis_df = recorder.load_object("portfolio_analysis/port_analysis_1day.pkl")

## analysis position

### report

In [ ]:
# 绘制收益/回撤等常用报告图
analysis_position.report_graph(report_normal_df)

### risk analysis

In [ ]:
# 绘制风险分析图（波动率、回撤等）
analysis_position.risk_analysis_graph(analysis_df, report_normal_df)

## analysis model

In [ ]:
# 从测试集提取真实标签（与 pred_df 合并用于 IC/分层分析）
label_df = dataset.prepare("test", col_set="label")
label_df.columns = ["label"]

### score IC

In [ ]:
# 合并真实标签与预测得分，并对齐索引后计算时序 IC
pred_label = pd.concat([label_df, pred_df], axis=1, sort=True).reindex(label_df.index)
analysis_position.score_ic_graph(pred_label)

### model performance

In [ ]:
# 绘制模型分层收益与表现图（打分分组的收益曲线等）
analysis_model.model_performance_graph(pred_label)